In [ ]:
# Build the class dataset: the full Parquet (benign + attack) converted to one CSV.
from pyspark.sql import SparkSession, functions as F

spark = (SparkSession.builder
         .master("local[*]")
         .appName("build_class_dataset")
         .config("spark.driver.memory", "8g")
         .getOrCreate())
spark.sparkContext.setLogLevel("ERROR")
spark.conf.set("spark.sql.session.timeZone", "UTC")

base = ("hdfs://192.168.1.250:9000/tmp/82913-CNT4416-202608/correlated_dataset/"
        "2026-09-26_00-28-45/finalized_data")
out  = "hdfs://192.168.1.250:9000/tmp/82913-CNT4416-202608/class_dataset_full_csv"

# Column order taken from the sample CSV header, so the file matches what the notebook expects
cols = spark.read.option("header", True).csv(base + "/Sample_Dataset_CSV/Benign/part-*.csv").columns

full = spark.read.parquet(base + "/Full_Dataset_Parquet/2026-*")
full = full.withColumn("datetime", F.date_format("datetime", "yyyy-MM-dd'T'HH:mm:ss.SSS'Z'"))
full = full.select([(F.col(c).cast("string") if c in full.columns
                     else F.lit(None).cast("string")).alias(c) for c in cols])

print("Total rows:", full.count())
full.groupBy("label_tactic").count().orderBy("count", ascending=False).show(truncate=False)

full.coalesce(1).write.mode("overwrite").option("header", True).csv(out)
print("Written to", out)

In [ ]:
# pull attack output to compare to mission logs
from pyspark.sql import SparkSession, functions as F
spark = SparkSession.builder.master("local[*]").config("spark.driver.memory", "8g").getOrCreate()
spark.sparkContext.setLogLevel("ERROR")
spark.conf.set("spark.sql.session.timeZone", "UTC")

df = (spark.read.option("header", True)
      .csv("hdfs://192.168.1.250:9000/tmp/82913-CNT4416-202608/class_dataset_full_csv"))
summary = (df.filter((F.col("label_tactic") != "none") & (F.col("label_technique") != "Duplicate"))
             .withColumn("day", F.substring("datetime", 1, 10))
             .groupBy("src_ip_zeek", "dest_ip_zeek", "label_technique", "label_tactic", "day")
             .agg(F.count(F.lit(1)).alias("connections"),
                  F.countDistinct("dest_port_zeek").alias("distinct_dest_ports"),
                  F.min("datetime").alias("first"), F.max("datetime").alias("last"))
             .orderBy("first"))
pdf = summary.toPandas()
pdf.to_csv("/home/ubuntu/CNT4416_attack_summary.csv", index=False)
print(len(pdf), "rows")
#

In [ ]:
#test missing entries
from datetime import datetime, timedelta
from pyspark.sql import SparkSession, functions as F

spark = SparkSession.builder.master("local[*]").config("spark.driver.memory", "6g").getOrCreate()
spark.sparkContext.setLogLevel("ERROR")
spark.conf.set("spark.sql.session.timeZone", "UTC")

logs = "hdfs://192.168.1.250:9000/tmp/82913-CNT4416-202608/nsm/zeek/logs"

checks = [  # (log line, source, destination IP or subnet prefix, start, end)
    (53, "143.88.5.10", "143.88.6.11", "2026-09-17T03:20:53Z", "2026-09-17T03:41:54Z"),  # CONTROL
    (20, "143.88.3.10", "143.88.3.12", "2026-09-05T12:36:08Z", "2026-09-05T12:36:15Z"),
    (21, "143.88.1.10", "143.88.1.11", "2026-09-08T14:40:00Z", "2026-09-08T15:05:00Z"),
    (22, "143.88.4.13", "143.88.4.11", "2026-09-06T14:12:00Z", "2026-09-06T14:12:01Z"),
    (29, "143.88.3.10", "143.88.4.11", "2026-09-09T13:59:00Z", "2026-09-09T13:59:10Z"),
    (30, "143.88.3.10", "143.88.4.12", "2026-09-09T14:41:00Z", "2026-09-09T14:42:20Z"),
    (31, "143.88.6.10", "143.88.7.",   "2026-09-09T16:14:00Z", "2026-09-09T16:16:00Z"),
    (50, "143.88.4.10", "143.88.5.11", "2026-09-17T01:35:53Z", "2026-09-17T01:35:53Z"),
    (51, "143.88.7.10", "143.88.8.11", "2026-09-17T22:03:47Z", "2026-09-17T22:04:01Z"),
    (52, "143.88.7.10", "143.88.8.12", "2026-09-17T22:15:53Z", "2026-09-17T22:16:06Z"),
    (59, "143.88.3.10", "143.88.4.11", "2026-09-23T13:25:47Z", "2026-09-23T13:26:03Z"),
    (64, "143.88.6.10", "143.88.7.11", "2026-09-23T21:50:00Z", "2026-09-23T21:50:00Z"),
    (65, "143.88.6.10", "143.88.7.12", "2026-09-23T22:40:00Z", "2026-09-23T22:40:00Z"),
]
def epoch(s): return datetime.fromisoformat(s.replace("Z", "+00:00")).timestamp()

# Only the day folders that exist for these entries (the day before through the day after)
days = sorted({(datetime.fromisoformat(s[:10]) + timedelta(days=k)).strftime("%Y-%m-%d")
               for _, _, _, s, _ in checks for k in (-1, 0, 1)})
jvm, conf = spark._jvm, spark._jsc.hadoopConfiguration()
fs = jvm.org.apache.hadoop.fs.Path(logs).getFileSystem(conf)
paths = [f"{logs}/{d}/conn.*.log.gz" for d in days
         if fs.globStatus(jvm.org.apache.hadoop.fs.Path(f"{logs}/{d}/conn.*.log.gz"))]
print("Reading", len(paths), "day folders")

conn = (spark.read.json(paths)
        .select(F.col("`id.orig_h`").alias("src"), F.col("`id.resp_h`").alias("dst"),
                F.col("`id.resp_p`").alias("dport"),
                F.coalesce(F.col("ts").cast("double"), F.to_timestamp("ts").cast("double")).alias("ts"))
        .cache())

windows = [("as logged (±2 min)", 0, 120), ("+5 h (local as UTC)", 5 * 3600, 120),
           ("whole day (±12 h)", 0, 12 * 3600)]
for line, src, dst, start, end in checks:
    dmatch = F.col("dst").startswith(dst) if dst.endswith(".") else (F.col("dst") == dst)
    pair = conn.filter((F.col("src") == src) & dmatch)
    print(f"\nLine {line}: {src} -> {dst}  logged {start}")
    for name, shift, pad in windows:
        r = (pair.filter(F.col("ts").between(epoch(start) + shift - pad, epoch(end) + shift + pad))
                 .agg(F.count(F.lit(1)).alias("n"),
                      F.array_sort(F.collect_set("dport")).alias("ports"),
                      F.from_unixtime(F.min("ts").cast("long")).alias("first"))
                 .first())
        ports = ", ".join(str(p) for p in (r["ports"] or [])[:8])
        print(f"  {name:22s} {r['n']:>6} conns  ports={ports}  first={r['first']}")
    any_src = conn.filter((F.col("src") == src)
                          & F.col("ts").between(epoch(start) - 43200, epoch(end) + 43200)).count()
    print(f"  {'any traffic from src':22s} {any_src:>6} conns that day (to anywhere)")